# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KMSiam/Flyrank-Internship-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

### Signal Audits Before Building the Rule

Before encoding a baseline heuristic, we audit two underlying signals using bucket tables with sample counts ($n$) displayed.

#### Signal 1 (Flag-linked: Staleness behind refresh flags)
- **Claim:** Content items that have not been updated for 90+ days (`days_since_last_update >= 90`) exhibit a higher rate of search decay (`is_declining_label == 1`).
- **Audit Table:** Binned by freshness tier (`0-30d`, `31-90d`, `91-180d`, `181+d`).
- **Verdict:** **MIXED**. Between 0–30 days and 91–180 days, the decline rate increases from 51.1% to 61.1% (+10 percentage points, $n = 9,171$). However, for pages older than 180 days ($n = 174$), the decline rate drops to 47.1% because older retained pages have low impression volume (median 15.5) and have already completed their decay curve (survivorship bias).

#### Signal 2 (Flag-linked: CTR-vs-Position behind CTR-fix logic)
- **Claim:** Content in striking distance (average SERP position 4–15) with below-median CTR declines faster than pages earning expected/above-median CTR.
- **Audit Table:** Filtered to striking distance ($4 \le \text{avg\_position} \le 15$), comparing below-median CTR vs above-median CTR ($n = 7,614$ vs $n = 7,723$).
- **Verdict:** **CONFIRMED**. Below-median CTR items decline at 61.33% versus 54.80% for above-median CTR items (+6.53 percentage points higher decay risk).

---

### The Baseline Rule in Plain Words

> **Rule:** *"A page is prioritized for a refresh action if it has proven search exposure (`impressions_90d >= 500`) and has entered the staleness decay window (`days_since_last_update >= 90`). Its priority score scales directly with its impression volume so editorial resources protect maximum potential traffic."*

- **Score Formula:**
  $$\text{baseline\_score} = (\text{days\_since\_last\_update} \ge 90) \times (\text{impressions\_90d} \ge 500) \times \text{impressions\_90d}$$
- **Reason Code:** `stale_high_visibility` (assigned to any item meeting both criteria; `insufficient_signal` otherwise).
- **Action Label:** `refresh_content` (assigned to prioritized items; `monitor` otherwise).


In [1]:
import os, json
import numpy as np
import pandas as pd

# Load Lane 2 starter dataset
data_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path) and os.path.exists('../' + data_path):
    data_path = '../' + data_path

df = pd.read_csv(data_path)
print(f"Loaded dataset: {len(df):,} rows, {len(df.columns)} columns")

# Target label: observed 90-day search decline
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
base_rate = float(df['is_declining_label'].mean())
print(f"Overall Dataset Decline Base Rate: {base_rate:.2%}\n")

# -------------------------------------------------------------
# Signal 1 Audit: Staleness vs Decline Rate
# -------------------------------------------------------------
print("=== Signal 1: Freshness Tier vs Decline Rate (Flag-linked) ===")
signal1_table = df.groupby('freshness_tier', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_count=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean'),
    median_impressions=('impressions_90d', 'median')
).reset_index()
signal1_table['decline_rate_pct'] = (signal1_table['decline_rate'] * 100).round(2)
display(signal1_table[['freshness_tier', 'n', 'declining_count', 'decline_rate_pct', 'median_impressions']])
print("Verdict: MIXED — 91-180d shows peak decay (61.11%), but 181+d drops to 47.13% due to low-volume survivorship bias.\n")

# -------------------------------------------------------------
# Signal 2 Audit: CTR Deficit in Striking Distance vs Decline Rate
# -------------------------------------------------------------
print("=== Signal 2: Striking-Distance CTR Deficit vs Decline Rate (CTR-fix logic) ===")
striking_df = df[(df['avg_position'] >= 4.0) & (df['avg_position'] <= 15.0)].copy()
median_ctr = striking_df['ctr'].median()

striking_df['ctr_category'] = np.where(
    striking_df['ctr'] < median_ctr, 
    'below_median_ctr (CTR fix candidate)', 
    'above_or_equal_median_ctr'
)

signal2_table = striking_df.groupby('ctr_category', observed=False).agg(
    n=('is_declining_label', 'count'),
    declining_count=('is_declining_label', 'sum'),
    decline_rate=('is_declining_label', 'mean'),
    median_position=('avg_position', 'median')
).reset_index()
signal2_table['decline_rate_pct'] = (signal2_table['decline_rate'] * 100).round(2)
display(signal2_table[['ctr_category', 'n', 'declining_count', 'decline_rate_pct', 'median_position']])
print(f"Verdict: CONFIRMED — Below-median CTR items decline at {signal2_table.loc[0, 'decline_rate_pct']}% vs {signal2_table.loc[1, 'decline_rate_pct']}% (+6.53 percentage points higher risk).\n")


Loaded dataset: 30,000 rows, 44 columns
Overall Dataset Decline Base Rate: 54.21%

=== Signal 1: Freshness Tier vs Decline Rate (Flag-linked) ===


,freshness_tier,n,declining_count,decline_rate_pct,median_impressions
0,0-30,20480,10473,51.14,470.0
1,181+,174,82,47.13,15.5
2,31-90,175,103,58.86,510.0
3,91-180,9171,5604,61.11,1692.0


Verdict: MIXED — 91-180d shows peak decay (61.11%), but 181+d drops to 47.13% due to low-volume survivorship bias.

=== Signal 2: Striking-Distance CTR Deficit vs Decline Rate (CTR-fix logic) ===


,ctr_category,n,declining_count,decline_rate_pct,median_position
0,above_or_equal_median_ctr,7723,4232,54.80,7.5
1,below_median_ctr (CTR fix candidate),7614,4670,61.33,8.2


Verdict: CONFIRMED — Below-median CTR items decline at 54.8% vs 61.33% (+6.53 percentage points higher risk).



## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

### Rule Implementation and Metrics Evaluation
We compute the transparent heuristic score without fitted parameters, attach the reason code and action label, rank all items, evaluate Precision@K against the base rate, and write the queue to `work/outputs/baseline_action_score.csv`.


In [2]:
# Encode the transparent rule
stale = (df['days_since_last_update'] >= 90).astype(int)
visible = (df['impressions_90d'] >= 500).astype(int)

df['baseline_score'] = stale * visible * df['impressions_90d']
df['reason_code'] = np.where(df['baseline_score'] > 0, 'stale_high_visibility', 'insufficient_signal')
df['action_label'] = np.where(df['baseline_score'] > 0, 'refresh_content', 'monitor')

# Sort descending to build the ranked queue
queue = df.sort_values(by='baseline_score', ascending=False).reset_index(drop=True)
queue['baseline_rank'] = queue.index + 1

# Precision@K function
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return float(np.asarray(labels)[order[:k]].mean())

p10 = precision_at_k(df['baseline_score'], df['is_declining_label'], 10)
p20 = precision_at_k(df['baseline_score'], df['is_declining_label'], 20)
p50 = precision_at_k(df['baseline_score'], df['is_declining_label'], 50)

print(f"=== Baseline Performance vs Base Rate ===")
print(f"Positive Base Rate : {base_rate:.2%}")
print(f"Precision@10       : {p10:.2%}")
print(f"Precision@20       : {p20:.2%}")
print(f"Precision@50       : {p50:.2%}\n")

# Ensure output directory exists
os.makedirs('work/outputs', exist_ok=True)
csv_out = 'work/outputs/baseline_action_score.csv'
json_out = 'work/outputs/baseline_metadata.json'

output_cols = [
    'content_id', 'client_id', 'baseline_rank', 'baseline_score',
    'reason_code', 'action_label', 'impressions_90d', 'clicks_90d',
    'days_since_last_update', 'avg_position', 'ctr', 'is_declining_label'
]

queue[output_cols].to_csv(csv_out, index=False)
print(f"✔ Successfully written {len(queue):,} rows to {csv_out}")

# Save metrics receipts JSON (committed to git)
metadata = {
    "rows": int(len(queue)),
    "base_rate": base_rate,
    "precision_at_10": p10,
    "precision_at_20": p20,
    "precision_at_50": p50,
    "rule_formula": "(days_since_last_update >= 90) * (impressions_90d >= 500) * impressions_90d",
    "top_score": float(queue['baseline_score'].max()),
    "items_prioritized": int((queue['baseline_score'] > 0).sum())
}
with open(json_out, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2)
print(f"✔ Successfully written metadata receipt to {json_out}")


=== Baseline Performance vs Base Rate ===
Positive Base Rate : 54.21%
Precision@10       : 60.00%
Precision@20       : 45.00%
Precision@50       : 44.00%

✔ Successfully written 30,000 rows to work/outputs/baseline_action_score.csv
✔ Successfully written metadata receipt to work/outputs/baseline_metadata.json


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

### Detailed Top-10 Audit (One Line Each)

1. **Rank 1 (`content_5fe46e04994d`):** Action: `refresh_content` | Why: 517,715 impressions with 741 clicks at position 4.2 after 104 days stale. | *What would make it wrong:* Ranking at pos 4.2 with low 0.14% CTR might reflect broad informational intent rather than actionable click demand.
2. **Rank 2 (`content_2dba2b1f9536`):** Action: `refresh_content` | Why: 443,434 impressions with 910 clicks, 104 days stale. | *What would make it wrong:* It is currently **stable** (`trend_direction = 'stable'`); refreshing a high-traffic stable asset risks destabilizing its existing rankings.
3. **Rank 3 (`content_2c2606c5d176`):** Action: `refresh_content` | Why: 347,399 impressions, 1,854 clicks, position 4.2, confirmed declining. | *What would make it wrong:* If the decline is seasonal search query demand rather than content quality decay.
4. **Rank 4 (`content_cb112fce36be`):** Action: `refresh_content` | Why: 309,910 impressions, position 5.6, confirmed declining. | *What would make it wrong:* If competitors added video/SERP features that cannibalized CTR regardless of content depth.
5. **Rank 5 (`content_9532f197bbc8`):** Action: `refresh_content` | Why: 309,192 impressions, 2,689 clicks, position 2.0, confirmed declining. | *What would make it wrong:* High-volume top-tier page where an aggressive rewrite could forfeit high brand keyword prominence.
6. **Rank 6 (`content_36ff89c8214e`):** Action: `refresh_content` | Why: 295,097 impressions, position 7.3, 104 days stale. | *What would make it wrong:* It is **stable** with low CTR (0.05%); ranking on page 1 for a high-volume head term with poor relevance may not yield clicks post-refresh.
7. **Rank 7 (`content_b28d1efd668f`):** Action: `refresh_content` | Why: 286,608 impressions on page 3 (`avg_pos = 26.2`). | *What would make it wrong:* It is **stable** and page 3 rank indicates weak domain authority on this topic, making on-page refresh insufficient.
8. **Rank 8 (`content_813e88069237`):** Action: `refresh_content` | Why: 233,561 impressions, position 26.2, confirmed declining. | *What would make it wrong:* Page 3 impressions are notoriously noisy impressions from bot scans or rare queries rather than true active user demand.
9. **Rank 9 (`content_c21024970297`):** Action: `refresh_content` | Why: 211,366 impressions, 870 clicks, position 5.1. | *What would make it wrong:* It is **stable**; prioritizing it over active decliners misallocates limited editorial capacity.
10. **Rank 10 (`content_c8e9d6ab9013`):** Action: `refresh_content` | Why: 208,678 impressions, 0 clicks, position 9.7, confirmed declining. | *What would make it wrong:* Earning 208k impressions with zero clicks indicates severe snippet misalignment or an accidental indexing anomaly rather than a content freshness issue.


In [3]:
# Display Top 20 Ranked Queue for Skeptical Review
top20 = queue.head(20).copy()
review_display_cols = [
    'baseline_rank', 'content_id', 'client_id', 'baseline_score', 
    'impressions_90d', 'clicks_90d', 'days_since_last_update', 
    'avg_position', 'ctr', 'trend_direction', 'is_declining_label'
]

print("Top 20 Review Table:")
display(top20[review_display_cols])

declining_in_top20 = int(top20['is_declining_label'].sum())
print(f"Top 20 Summary: {declining_in_top20}/20 ({declining_in_top20/20:.1%}) are actively declining.")


Top 20 Review Table:


,baseline_rank,content_id,client_id,baseline_score,impressions_90d,clicks_90d,days_since_last_update,avg_position,ctr,trend_direction,is_declining_label
0,1,content_5fe46e04994d,client_4e07408562,517715,517715,741,104,4.2,0.14,down,1
1,2,content_2dba2b1f9536,client_6208ef0f77,443434,443434,910,104,27.9,0.21,stable,0
2,3,content_2c2606c5d176,client_19581e27de,347399,347399,1854,104,4.2,0.53,down,1
3,4,content_cb112fce36be,client_19581e27de,309910,309910,492,104,5.6,0.16,down,1
4,5,content_9532f197bbc8,client_4e07408562,309192,309192,2689,104,2.0,0.87,down,1
5,6,content_36ff89c8214e,client_19581e27de,295097,295097,154,104,7.3,0.05,stable,0
6,7,content_b28d1efd668f,client_6208ef0f77,286608,286608,169,104,26.2,0.06,stable,0
7,8,content_813e88069237,client_6208ef0f77,233561,233561,129,104,26.2,0.06,down,1
8,9,content_c21024970297,client_19581e27de,211366,211366,870,104,5.1,0.41,stable,0
9,10,content_c8e9d6ab9013,client_19581e27de,208678,208678,0,104,9.7,0.00,down,1


Top 20 Summary: 9/20 (45.0%) are actively declining.


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

### Critical Review: Which Picks Look Wrong?

1. **Volume Domination Over Decline Reality:**
   Because the baseline rule ranks by raw `impressions_90d`, huge pages dominate the top ranks even when they are completely stable or growing:
   - **Rank 2 (`content_2dba2b1f9536`):** 443,434 impressions and 910 clicks, but `trend_direction == 'stable'`.
   - **Rank 6 (`content_36ff89c8214e`):** 295,097 impressions, but `trend_direction == 'stable'`.
   - **Rank 18 (`content_f02b48f88241`):** 181,514 impressions and 178 clicks, but `trend_direction == 'up'`. Recommending a content refresh on a growing page wastes resources and risks breaking an asset that Google is actively favoring!

2. **Client Concentration Bias:**
   In the top 20, 18 of the 20 rows belong to only two clients (`client_19581e27de` and `client_6208ef0f77`). High-volume enterprise domains drown out smaller clients in a pure impression-weighted rule.

3. **Page-3 False Hope:**
   Picks like Rank 7 (`avg_pos = 26.2`) have high impression exposure but rank deep on page 3. Refreshing content cannot overcome a domain authority deficit.

---

### Leakage and Guardrail Verification

- **No Future-Window Inputs:** The rule uses strictly trailing 90-day exposure (`impressions_90d`) and historical staleness (`days_since_last_update`).
- **No Label-Derived Features:** The rule does NOT read `trend_pct` or `trend_direction`.
- **No Product Decision Flags:** The rule does NOT use pre-existing proprietary health flags or internal scores.
- **CI Data Leak Guard:** `work/outputs/baseline_action_score.csv` is written to `work/outputs/` which is ignored by `.gitignore` (`work/**/*.csv`), ensuring zero bulk data files leak into git.


In [4]:
# Automated Leakage and Weak-Pick Audit
print("=== Automated Audit Checks ===")

# Check 1: Weak picks identification (stable or growing pages flagged as top picks)
weak_picks = top20[top20['is_declining_label'] == 0]
print(f"1. Weak Picks in Top 20 (False Positives): {len(weak_picks)} / 20")
for _, r in weak_picks.iterrows():
    print(f"   - Rank {int(r['baseline_rank'])} ({r['content_id']}): {r['trend_direction'].upper()} with {int(r['impressions_90d']):,} impressions")

# Check 2: Verify zero correlation with outcome-only features in score computation
formula_inputs = ['days_since_last_update', 'impressions_90d']
forbidden_inputs = ['trend_pct', 'trend_direction', 'is_declining_label']

for col in forbidden_inputs:
    assert col not in formula_inputs, f"LEAKAGE VIOLATION: {col} used in baseline formula!"
print("\n2. ✔ Zero leakage inputs: Baseline rule strictly uses historical pre-decision signals.")

# Check 3: Confirm output CSV exists on disk
assert os.path.exists('work/outputs/baseline_action_score.csv'), "Output CSV missing!"
csv_size = os.path.getsize('work/outputs/baseline_action_score.csv')
print(f"3. ✔ Output file confirmed: work/outputs/baseline_action_score.csv ({csv_size:,} bytes).")


=== Automated Audit Checks ===
1. Weak Picks in Top 20 (False Positives): 11 / 20
   - Rank 2 (content_2dba2b1f9536): STABLE with 443,434 impressions
   - Rank 6 (content_36ff89c8214e): STABLE with 295,097 impressions
   - Rank 7 (content_b28d1efd668f): STABLE with 286,608 impressions
   - Rank 9 (content_c21024970297): STABLE with 211,366 impressions
   - Rank 11 (content_b511d4bc4ad2): STABLE with 205,915 impressions
   - Rank 12 (content_d17681677e69): STABLE with 201,584 impressions
   - Rank 13 (content_a7427266c305): STABLE with 201,111 impressions
   - Rank 14 (content_c5063073d048): STABLE with 192,205 impressions
   - Rank 17 (content_33b4dceecad1): STABLE with 181,574 impressions
   - Rank 18 (content_f02b48f88241): UP with 181,514 impressions
   - Rank 20 (content_bb5bd5f771dc): STABLE with 176,296 impressions

2. ✔ Zero leakage inputs: Baseline rule strictly uses historical pre-decision signals.
3. ✔ Output file confirmed: work/outputs/baseline_action_score.csv (2,976,031 b

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
